Time series data and autocorrelation for radius of gyration R(g) and end to end distance (Ree)

In [ ]:
import os
import numpy as np
import gsd.hoomd
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ======================= Helper Functions =======================

def apply_pbc(positions, box):
    """Apply minimum image convention to positions."""
    Lx, Ly, Lz, xy, xz, yz = box
    box_lengths = np.array([Lx, Ly, Lz])
    positions = positions.copy()
    for dim in range(3):
        positions[:, dim] -= np.rint(positions[:, dim] / box_lengths[dim]) * box_lengths[dim]
    return positions

def autocorrelation(series):
    series = series - np.mean(series)
    ac = np.correlate(series, series, mode='full')
    ac = ac[ac.size // 2:]
    ac /= ac[0]
    return ac

def exp_decay(x, tau, A):
    return A * np.exp(-x / tau)

def end_to_end_autocorr_and_series(frames, Ca_index, n_chains, atoms_per_chain):
    R_ee_per_chain = []
    for frame in frames:
        pos = frame.particles.position
        types = frame.particles.typeid
        Ca_positions = pos[types == Ca_index]
        Ca_positions = apply_pbc(Ca_positions, frame.configuration.box)

        chain_R = []
        for i in range(n_chains):
            start = i * atoms_per_chain
            end = (i + 1) * atoms_per_chain
            delta = Ca_positions[end-1] - Ca_positions[start]

            # Apply PBC
            Lx, Ly, Lz, xy, xz, yz = frame.configuration.box
            box_lengths = np.array([Lx, Ly, Lz])
            delta -= np.rint(delta / box_lengths) * box_lengths

            chain_R.append(np.linalg.norm(delta))

        R_ee_per_chain.append(chain_R)

    R_ee_per_chain = np.array(R_ee_per_chain)
    ac_per_chain = np.array([autocorrelation(R_ee_per_chain[:, i]) for i in range(n_chains)])
    return R_ee_per_chain, ac_per_chain

def calculate_R_g_squared(positions, masses):
    com = np.sum(masses[:, None] * positions, axis=0) / np.sum(masses)
    squared_distances = np.sum(masses[:, None] * (positions - com)**2, axis=1)
    return np.sum(squared_distances) / np.sum(masses)

def compute_Rg_series(frames, n_chains):
    Rg_per_chain = []
    for frame in frames:
        positions = frame.particles.position.copy()
        positions = apply_pbc(positions, frame.configuration.box)
        masses = frame.particles.mass

        num_particles_per_chain = len(positions) // n_chains
        chain_Rg = []
        for i in range(n_chains):
            start = i * num_particles_per_chain
            end = (i + 1) * num_particles_per_chain
            Rg2 = calculate_R_g_squared(positions[start:end], masses[start:end])
            chain_Rg.append(np.sqrt(Rg2))

        Rg_per_chain.append(chain_Rg)

    Rg_per_chain = np.array(Rg_per_chain)
    ac_per_chain = np.array([autocorrelation(Rg_per_chain[:, i]) for i in range(n_chains)])
    return Rg_per_chain, ac_per_chain

# ======================= Trajectory Sets =======================

sets = [
    {"name": "Full_length", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/"},
    {"name": "H0_H3", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/3de12d848c06a70b5668c64369550562/"},
    {"name": "H4_H6", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/7fd5f713b9bd4167bafd3d2d9378bc51/"}
]

bias_values = np.linspace(0.75, 1.0, 12)
n_chains = 64

# ======================= Main Processing =======================

for s in sets:
    print(f"\n=== Processing set: {s['name']} ===")
    output_dir = os.path.join(os.getcwd(), s['name'])
    os.makedirs(output_dir, exist_ok=True)

    tau_R_ee_mean_list = []
    tau_R_ee_std_list = []
    tau_Rg_mean_list = []
    tau_Rg_std_list = []

    for bias in bias_values:
        trajectory_name = (
            os.path.join(s['base_path'], f"trajectory_bias_{bias}.gsd")
            if np.isclose(bias, 0.75) or np.isclose(bias, 1.0)
            else os.path.join(s['base_path'], f"trajectory_bias_{bias:.16f}.gsd")
        )

        if not os.path.isfile(trajectory_name):
            print(f"File {trajectory_name} not found. Skipping.")
            tau_R_ee_mean_list.append(np.nan)
            tau_R_ee_std_list.append(np.nan)
            tau_Rg_mean_list.append(np.nan)
            tau_Rg_std_list.append(np.nan)
            continue
        
        print(f"\nProcessing file: {trajectory_name}")
        
        traj = gsd.hoomd.open(trajectory_name, 'r')
        frames = traj[20:20000]  # skip initial frames
        all_types = frames[0].particles.types
        Ca_index = all_types.index("Ca")
        types = frames[0].particles.typeid
        total_Ca = np.count_nonzero(types == Ca_index)
        atoms_per_chain = total_Ca // n_chains

        # ---- End-to-End Distance ----
        R_ee_per_chain, ac_R_ee = end_to_end_autocorr_and_series(frames, Ca_index, n_chains, atoms_per_chain)

        # Store time-series & stats
        df_Ree = pd.DataFrame(R_ee_per_chain, columns=[f'chain_{i}' for i in range(n_chains)])
        df_Ree.to_csv(os.path.join(output_dir, f"R_ee_timeseries_bias_{bias}.csv"), index=False)

        df_Ree_stats = pd.DataFrame({
            "chain": [f"chain_{i}" for i in range(n_chains)],
            "mean_R_ee": np.nanmean(R_ee_per_chain, axis=0),
            "std_R_ee": np.nanstd(R_ee_per_chain, axis=0)
        })
        df_Ree_stats.to_csv(os.path.join(output_dir, f"R_ee_timeseries_stats_bias_{bias}.csv"), index=False)


        # Fit correlation times
        tau_ee_chains = []
        for ac in ac_R_ee:
            lag = np.arange(len(ac))
            mask = ac > 0
            try:
                popt, _ = curve_fit(exp_decay, lag[mask], ac[mask], p0=(len(lag)/10, 1.0))
                tau_ee_chains.append(popt[0])
            except:
                tau_ee_chains.append(np.nan)
        tau_ee_chains = np.array(tau_ee_chains)

        tau_R_ee_mean_list.append(np.nanmean(tau_ee_chains))
        tau_R_ee_std_list.append(np.nanstd(tau_ee_chains))

        pd.DataFrame(ac_R_ee.T, columns=[f'chain_{i}' for i in range(n_chains)]).to_csv(
            os.path.join(output_dir, f'end_to_end_autocorr_bias_{bias}.csv'), index=False
        )

        # ---- Radius of Gyration ----
        Rg_per_chain, ac_Rg = compute_Rg_series(frames, n_chains)

        # Store Rg time-series & stats
        df_Rg = pd.DataFrame(Rg_per_chain, columns=[f'chain_{i}' for i in range(n_chains)])
        df_Rg.to_csv(os.path.join(output_dir, f"Rg_timeseries_bias_{bias}.csv"), index=False)

        df_Rg_stats = pd.DataFrame({
            "chain": [f"chain_{i}" for i in range(n_chains)],
            "mean_Rg": np.nanmean(Rg_per_chain, axis=0),
            "std_Rg": np.nanstd(Rg_per_chain, axis=0)
        })
        df_Rg_stats.to_csv(os.path.join(output_dir, f"Rg_timeseries_stats_bias_{bias}.csv"), index=False)


        # Fit correlation times
        tau_rg_chains = []
        for ac in ac_Rg:
            lag = np.arange(len(ac))
            mask = ac > 0
            try:
                popt, _ = curve_fit(exp_decay, lag[mask], ac[mask], p0=(len(lag)/10, 1.0))
                tau_rg_chains.append(popt[0])
            except:
                tau_rg_chains.append(np.nan)

        tau_rg_chains = np.array(tau_rg_chains)

        tau_Rg_mean_list.append(np.nanmean(tau_rg_chains))
        tau_Rg_std_list.append(np.nanstd(tau_rg_chains))

        pd.DataFrame(ac_Rg.T, columns=[f'chain_{i}' for i in range(n_chains)]).to_csv(
            os.path.join(output_dir, f'Rg_autocorr_bias_{bias}.csv'), index=False
        )

        print(f"Bias {bias:.3f} processed: τ_R_ee={tau_R_ee_mean_list[-1]:.2f}±{tau_R_ee_std_list[-1]:.2f}, "
              f"τ_Rg={tau_Rg_mean_list[-1]:.2f}±{tau_Rg_std_list[-1]:.2f}")

    # ---- Save summary CSV ----
    summary_df = pd.DataFrame({
        'Bias': bias_values,
        'tau_R_ee_mean': tau_R_ee_mean_list,
        'tau_R_ee_std': tau_R_ee_std_list,
        'tau_Rg_mean': tau_Rg_mean_list,
        'tau_Rg_std': tau_Rg_std_list
    })
    summary_df.to_csv(os.path.join(output_dir, 'correlation_times_vs_bias.csv'), index=False)

    # ---- Plot with error bars ----
    plt.figure(figsize=(7,5))
    plt.errorbar(bias_values, tau_R_ee_mean_list, yerr=tau_R_ee_std_list, fmt='o-', label='τ R_ee', capsize=5)
    plt.errorbar(bias_values, tau_Rg_mean_list, yerr=tau_Rg_std_list, fmt='s-', label='τ Rg', capsize=5)
    plt.xlabel('Bias')
    plt.ylabel('Correlation time τ (frames)')
    plt.title(f'Correlation Time vs Bias ({s["name"]})')
    plt.grid(True, alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'correlation_time_vs_bias.png'), dpi=300)
    plt.close()


Correlation Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load CSV
file_path = '/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/Full_length/correlation_times_vs_bias.csv'
data = pd.read_csv(file_path)

# Convert Bias to %
data['Bias_percent'] = data['Bias'] * 100

# Convert to microseconds
data['tau_R_ee_mean_us'] = data['tau_R_ee_mean'] / 1000
data['tau_R_ee_std_us'] = data['tau_R_ee_std'] / 1000
data['tau_Rg_mean_us'] = data['tau_Rg_mean'] / 1000
data['tau_Rg_std_us'] = data['tau_Rg_std'] / 1000

# If you want SEM instead of std, use this:
data['tau_R_ee_std_us'] /= np.sqrt(12)
data['tau_Rg_std_us'] /= np.sqrt(12)

plt.figure(figsize=(10,7))

# τ_Ree
plt.errorbar(
    data['Bias_percent'],
    data['tau_R_ee_mean_us'],
    yerr=data['tau_R_ee_std_us'],
    fmt='o-',
    markersize=7,
    linewidth=2.5,
    elinewidth=1.8,
    capsize=4,
    color='red',
    label=r'$\tau_{R_{ee}}$'
)

# τ_Rg
plt.errorbar(
    data['Bias_percent'],
    data['tau_Rg_mean_us'],
    yerr=data['tau_Rg_std_us'],
    fmt='s--',
    markersize=7,
    linewidth=2.5,
    elinewidth=1.8,
    capsize=4,
    color='darkred',
    label=r'$\tau_{R_g}$'
)

# Labels
plt.xlabel('H-bond Strength (%)', fontsize=30, fontweight='bold')
plt.ylabel('Correlation Time (µs)', fontsize=30, fontweight='bold')

# Title
#plt.title('Correlation Time vs H-bond Strength',fontsize=30, fontweight='bold')

# Ticks
plt.xticks(fontsize=26, fontweight='bold')
plt.yticks(fontsize=26, fontweight='bold')

# Bold legend with title
legend = plt.legend(
    title='IM30',
    title_fontsize=20,
    fontsize=20,
    frameon=False
)
for text in legend.get_texts():
    text.set_fontweight('bold')
legend.get_title().set_fontweight('bold')

# Thick border box
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

# Save
save_path = '/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/correlation_times_vs_bias_styled.png'
plt.savefig(save_path, dpi=600)

plt.show()

print(f"Figure saved to: {save_path}")


Time Series Plot of R_g

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

# ---- DIRECTORIES ----
dirs = {
    "IM30": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/Full_length/",
    "IM30 H0-3": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H0_H3/",
    "IM30 H4-6": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H4_H6/"
}

for region_label, data_dir in dirs.items():
    # Find all CSVs for this region
    file_pattern = os.path.join(data_dir, "Rg_timeseries_bias_*.csv")
    files = sorted(glob.glob(file_pattern))
    print(f"{region_label}: Found {len(files)} files.")

    plt.figure(figsize=(12, 8))

    for f in files:
        # Extract bias
        bias_str = os.path.basename(f).replace("Rg_timeseries_bias_", "").replace(".csv", "")
        bias_val = float(bias_str)

        # Load CSV
        df = pd.read_csv(f)
        df = df.iloc[100:].reset_index(drop=True)

        # Identify chain columns
        chain_cols = [c for c in df.columns if c.startswith("chain_")]

        # Mean & std across chains, divide std by sqrt(12)
        mean_series = df[chain_cols].mean(axis=1)
        std_series = df[chain_cols].std(axis=1) / np.sqrt(12)

        time = mean_series.index / 1000.0  # convert to μs

        # Plot mean curve (default color cycle)
        plt.plot(time, mean_series, label=f"{bias_val*100:.0f}%")

        # Shaded ± std
        plt.fill_between(time,
                         mean_series - std_series,
                         mean_series + std_series,
                         alpha=0.25)

    # Plot formatting
    plt.xlabel("Time (μs)", fontsize=30, fontweight="bold")
    plt.ylabel("Rg (Å)", fontsize=30, fontweight="bold")
    plt.title(region_label, fontsize=30, fontweight="bold")
    plt.xticks(fontsize=26, fontweight="bold")
    plt.yticks(fontsize=26, fontweight="bold")

    # Legend bold
    legend = plt.legend(title="H-bond Strength", fontsize=26, title_fontsize=26, ncol=2)
    for text in legend.get_texts():
        text.set_fontweight("bold")
    legend.get_title().set_fontweight("bold")

    plt.tight_layout()

    # Save figure
    save_name = f"{region_label.replace(' ', '_')}_Rg_timeseries_bold.png"
    plt.savefig(save_name, dpi=300)
    print(f"Saved figure: {save_name}")

    plt.show()


Radius of gyration vs bias HBS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

# ---- DIRECTORIES ----
dirs = {
    "IM30": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/Full_length/",
    "IM30 H0-3": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H0_H3/",
    "IM30 H4-6": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H4_H6/"
}

# ---- OUTPUT FILE ----
save_path = "Rg_vs_bias_all_regions_styled.png"

# Colors for the curves
colors = {
    "IM30": "red",
    "IM30 H0-3": "green",
    "IM30 H4-6": "blue"
}

plt.figure(figsize=(10,7))

for label, directory in dirs.items():
    file_pattern = os.path.join(directory, "Rg_timeseries_bias_*.csv")
    files = sorted(glob.glob(file_pattern))
    print(f"{label}: found {len(files)} files")

    bias_vals = []
    avg_Rg_vals = []
    std_Rg_vals = []

    for f in files:
        # Extract bias value
        bias_str = os.path.basename(f).replace("Rg_timeseries_bias_", "").replace(".csv", "")
        bias_val = float(bias_str)
        bias_vals.append(bias_val)

        # Load data
        df = pd.read_csv(f)

        # Remove first 10,000 frames
        df = df.iloc[10000:].reset_index(drop=True)

        # Identify chain columns
        chain_cols = [c for c in df.columns if c.startswith("chain_")]

        # Rg mean(t) across chains
        mean_series = df[chain_cols].mean(axis=1)

        # Average Rg and std across time
        avg_Rg_vals.append(mean_series.mean())
        std_Rg_vals.append(mean_series.std() / np.sqrt(12))  # SEM

    # Convert bias → H-bond strength %
    bias_percent = [b * 100 for b in bias_vals]

    # ---- PLOT each dataset ----
    plt.errorbar(
        bias_percent,
        avg_Rg_vals,
        yerr=std_Rg_vals,
        fmt="o-",
        color=colors[label],
        capsize=5,
        linewidth=2.5,
        markersize=7,
        label=label
    )

# ---- FINAL PLOT DETAILS ----
plt.xlabel("H-bond Strength (%)", fontweight="bold", fontsize=30)
plt.ylabel("Rg (Å)", fontweight="bold", fontsize=30)
plt.xticks(fontsize=26, fontweight="bold")
plt.yticks(fontsize=26, fontweight="bold")

# Bold legend
legend = plt.legend(title="", fontsize=26)
for text in legend.get_texts():
    text.set_fontweight('bold')

# Thick axes box
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

# ---- SAVE FIGURE ----
plt.savefig(save_path, dpi=600)
print(f"Saved figure to: {save_path}")

plt.show()


Time series for Ree

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

# ---- DIRECTORIES ----
dirs = {
    "IM30": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/Full_length/",
    "IM30 H0-3": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H0_H3/",
    "IM30 H4-6": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H4_H6/"
}

for region_label, data_dir in dirs.items():
    # Find all CSVs for this region
    file_pattern = os.path.join(data_dir, "R_ee_timeseries_bias_*.csv")
    files = sorted(glob.glob(file_pattern))
    print(f"{region_label}: Found {len(files)} files.")

    plt.figure(figsize=(12, 8))

    for f in files:
        # Extract bias
        bias_str = os.path.basename(f).replace("R_ee_timeseries_bias_", "").replace(".csv", "")
        bias_val = float(bias_str)

        # Load CSV and skip first 100 frames
        df = pd.read_csv(f)
        df = df.iloc[100:].reset_index(drop=True)

        # Chain columns: chain_0 ... chain_63
        chain_cols = [col for col in df.columns if col.startswith("chain_")]

        # Mean & std across chains, divide std by sqrt(12)
        mean_series = df[chain_cols].mean(axis=1)
        std_series = df[chain_cols].std(axis=1) / np.sqrt(12)

        time = mean_series.index / 1000.0  # convert to μs

        # Plot mean curve using default color cycle
        plt.plot(time, mean_series, label=f"{bias_val*100:.0f}%")

        # Shaded std region using same color as line
        plt.fill_between(
            time,
            mean_series - std_series,
            mean_series + std_series,
            alpha=0.25
        )

    # Plot formatting with bold fonts
    plt.xlabel("Time (μs)", fontsize=30, fontweight="bold")
    plt.ylabel("Rₑₑ (Å)", fontsize=30, fontweight="bold")
    plt.title(region_label, fontsize=30, fontweight="bold")
    plt.xticks(fontsize=26, fontweight="bold")
    plt.yticks(fontsize=26, fontweight="bold")
    plt.ylim(0, None)

    # Legend bold
    legend = plt.legend(title="H-bond Strength", fontsize=26, title_fontsize=26, ncol=2)
    for text in legend.get_texts():
        text.set_fontweight("bold")
    legend.get_title().set_fontweight("bold")

    plt.tight_layout()

    # Save figure
    save_name = f"{region_label.replace(' ', '_')}_Ree_timeseries_default_color_bold.png"
    plt.savefig(save_name, dpi=600)
    print(f"Saved figure: {save_name}")

    plt.show()


R_ee vs HBS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

# ---- DIRECTORIES ----
dirs = {
    "IM30": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/Full_length/",
    "IM30 H0-3": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H0_H3/",
    "IM30 H4-6": "/home/POLY/bhandarit/Desktop/chain_n_chain/End_end_distance/Ree_with_time/autocorrelation/H4_H6/"
}

# ---- OUTPUT FILE ----
save_path = "Ree_vs_bias_all_regions_styled.png"

# Colors for the curves
colors = {
    "IM30": "red",
    "IM30 H0-3": "green",
    "IM30 H4-6": "blue"
}

plt.figure(figsize=(10,7))

for label, directory in dirs.items():
    file_pattern = os.path.join(directory, "R_ee_timeseries_bias_*.csv")
    files = sorted(glob.glob(file_pattern))
    print(f"{label}: found {len(files)} files")

    bias_vals = []
    avg_Ree_vals = []
    std_Ree_vals = []

    for f in files:
        # Extract numeric bias value
        bias_str = os.path.basename(f).replace("R_ee_timeseries_bias_", "").replace(".csv", "")
        bias_val = float(bias_str)
        bias_vals.append(bias_val)

        # Load CSV
        df = pd.read_csv(f)

        # Remove first 10,000 frames
        df = df.iloc[10000:].reset_index(drop=True)

        # Identify chain columns
        chain_cols = [c for c in df.columns if c.startswith("chain_")]
        N_chains = len(chain_cols)

        # Mean across chains at each timestep
        mean_series = df[chain_cols].mean(axis=1)

        # Compute average and SEM across time
        avg_Ree_vals.append(mean_series.mean())
        std_Ree_vals.append(mean_series.std() / np.sqrt(12))

    # Convert bias → H-bond %
    bias_percent = [b * 100 for b in bias_vals]

    # Plot with error bars
    plt.errorbar(
        bias_percent,
        avg_Ree_vals,
        yerr=std_Ree_vals,
        fmt="o-",
        color=colors[label],
        capsize=5,
        linewidth=2.5,
        markersize=7,
        label=label
    )

# ---- FINAL PLOT DETAILS ----
plt.xlabel("H-bond Strength (%)", fontweight="bold", fontsize=30)
plt.ylabel("Rₑₑ (Å)", fontweight="bold", fontsize=30)
plt.xticks(fontsize=26, fontweight='bold')
plt.yticks(fontsize=26, fontweight='bold')

# Bold legend
legend = plt.legend(title="", fontsize=26)
for text in legend.get_texts():
    text.set_fontweight('bold')

# Thick axes box
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

# ---- SAVE FIGURE ----
plt.savefig(save_path, dpi=600)
print(f"Saved figure to: {save_path}")

plt.show()
